# Sentiment Analysis using BERT — IMDb Movie Reviews

**Case Study:** Sentiment Analysis using BERT  
**Dataset:** IMDb Large Movie Review Dataset (ACL IMDb folder)

In [1]:
# # If required, install the libraries.
# # Run this cell once.

# !pip install -q transformers datasets accelerate scikit-learn pandas numpy matplotlib seaborn torch

In [ ]:
# pip install tf-keras

  Using cached tf_keras-2.21.0-py3-none-any.whl.metadata (1.8 kB)
  Using cached tensorflow-2.21.0-cp312-cp312-win_amd64.whl.metadata (4.5 kB)
  Using cached flatbuffers-25.12.19-py2.py3-none-any.whl.metadata (1.0 kB)
  Using cached protobuf-7.36.2-cp310-abi3-win_amd64.whl.metadata (595 bytes)
  Using cached keras-3.15.1-py3-none-any.whl.metadata (6.4 kB)
Using cached tf_keras-2.21.0-py3-none-any.whl (1.7 MB)
   ---------------------------------------- 0.0/350.9 MB ? eta -:--:--
   ---------------------------------------- 3.1/350.9 MB 15.3 MB/s eta 0:00:23
    --------------------------------------- 8.1/350.9 MB 20.1 MB/s eta 0:00:18
   - -------------------------------------- 12.1/350.9 MB 19.3 MB/s eta 0:00:18
   - -------------------------------------- 16.5/350.9 MB 20.0 MB/s eta 0:00:17
   -- ------------------------------------- 19.7/350.9 MB 18.8 MB/s eta 0:00:18
   -- ------------------------------------- 24.1/350.9 MB 19.6 MB/s eta 0:00:17
   --- -------------------------------

  You can safely remove it manually.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
databricks-sdk 0.118.0 requires protobuf!=5.26.*,!=5.27.*,!=5.28.*,!=5.29.0,!=5.29.1,!=5.29.2,!=5.29.3,!=5.29.4,!=6.30.0,!=6.30.1,!=6.31.0,<7.0,>=4.25.8, but you have protobuf 7.36.2 which is incompatible.
googleapis-common-protos 1.70.0 requires protobuf!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<7.0.0,>=3.20.2, but you have protobuf 7.36.2 which is incompatible.
google-api-core 2.25.1 requires protobuf!=3.20.0,!=3.20.1,!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<7.0.0,>=3.19.5, but you have protobuf 7.36.2 which is incompatible.
google-cloud-aiplatform 1.111.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,

In [4]:
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

print("Transformers imported successfully!")

ValueError: Your currently installed version of Keras is Keras 3, but this is not yet supported in Transformers. Please install the backwards-compatible tf-keras package with `pip install tf-keras`.

## 1. Import Libraries

In [2]:
import os
import re
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

ValueError: Your currently installed version of Keras is Keras 3, but this is not yet supported in Transformers. Please install the backwards-compatible tf-keras package with `pip install tf-keras`.

## 2. Set Dataset Path

The code first checks whether the `acllmdb` folder is in the current working directory.

If your folder is somewhere else, change `DATA_DIR` manually.

Examples:

```python
DATA_DIR = "./acllmdb"
```

or on Windows:

```python
DATA_DIR = r"C:\Users\YourName\Downloads\acllmdb"
```

In [ ]:
# Change this only if your acllmdb folder is somewhere else.
DATA_DIR = "./acllmdb"

print("Current working directory:", os.getcwd())
print("Dataset path:", os.path.abspath(DATA_DIR))
print("Dataset exists:", os.path.exists(DATA_DIR))

assert os.path.exists(DATA_DIR), (
    "acllmdb folder not found. Put this notebook beside the acllmdb folder "
    "or change DATA_DIR to the correct path."
)

## 3. Read IMDb Reviews

Each `.txt` file contains one movie review.

- `pos` → positive sentiment → label `1`
- `neg` → negative sentiment → label `0`

In [ ]:
def load_reviews(folder, label):
    rows = []

    if not os.path.exists(folder):
        raise FileNotFoundError(f"Folder not found: {folder}")

    for filename in os.listdir(folder):
        if filename.endswith(".txt"):
            filepath = os.path.join(folder, filename)

            try:
                with open(filepath, "r", encoding="utf-8") as f:
                    text = f.read().strip()
            except UnicodeDecodeError:
                with open(filepath, "r", encoding="latin-1") as f:
                    text = f.read().strip()

            rows.append({
                "review": text,
                "label": label
            })

    return rows


train_pos = load_reviews(os.path.join(DATA_DIR, "train", "pos"), 1)
train_neg = load_reviews(os.path.join(DATA_DIR, "train", "neg"), 0)

test_pos = load_reviews(os.path.join(DATA_DIR, "test", "pos"), 1)
test_neg = load_reviews(os.path.join(DATA_DIR, "test", "neg"), 0)

train_data = train_pos + train_neg
test_data = test_pos + test_neg

train_df = pd.DataFrame(train_data)
test_df = pd.DataFrame(test_data)

# Shuffle the rows
train_df = train_df.sample(frac=1, random_state=42).reset_index(drop=True)
test_df = test_df.sample(frac=1, random_state=42).reset_index(drop=True)

print("Training samples:", len(train_df))
print("Testing samples :", len(test_df))
print("\nTraining label distribution:")
print(train_df["label"].value_counts())

In [ ]:
# Display sample reviews
display(train_df.head())

print("\nPositive review example:")
print(train_df[train_df["label"] == 1]["review"].iloc[0][:1000])

print("\nNegative review example:")
print(train_df[train_df["label"] == 0]["review"].iloc[0][:1000])

## 4. Basic Data Exploration

In [ ]:
print("Training shape:", train_df.shape)
print("Testing shape :", test_df.shape)

print("\nMissing values:")
print(train_df.isnull().sum())

print("\nDuplicate reviews:", train_df["review"].duplicated().sum())

train_df["review_length"] = train_df["review"].str.len()

print("\nReview length statistics:")
print(train_df["review_length"].describe())

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=train_df, x="label")
plt.title("IMDb Sentiment Distribution")
plt.xlabel("Sentiment (0 = Negative, 1 = Positive)")
plt.ylabel("Number of Reviews")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(
    data=train_df,
    x="review_length",
    hue="label",
    bins=50,
    element="step"
)
plt.title("Review Length Distribution")
plt.xlabel("Review length")
plt.show()

## 5. Prepare the Data

BERT performs its own tokenization, so we should **not** use TF-IDF, Bag-of-Words, LabelEncoder, or StandardScaler for the review text.

We only need:

- review text
- sentiment label

In [ ]:
# Remove empty reviews if any.
# This does NOT remove normal rows; it only removes rows with no usable text.
train_df = train_df[train_df["review"].str.strip().ne("")].reset_index(drop=True)
test_df = test_df[test_df["review"].str.strip().ne("")].reset_index(drop=True)

# Keep only required columns
train_df = train_df[["review", "label"]]
test_df = test_df[["review", "label"]]

print("Training shape:", train_df.shape)
print("Testing shape :", test_df.shape)

## 6. Create Validation Set

The official IMDb test set is kept untouched for final evaluation.

We split the official training set into:

- 90% training
- 10% validation

In [ ]:
train_texts, val_texts, train_labels, val_labels = train_test_split(
    train_df["review"].tolist(),
    train_df["label"].tolist(),
    test_size=0.10,
    random_state=42,
    stratify=train_df["label"]
)

test_texts = test_df["review"].tolist()
test_labels = test_df["label"].tolist()

print("Training:", len(train_texts))
print("Validation:", len(val_texts))
print("Testing:", len(test_texts))

## 7. Optional Dataset Size Control

Full IMDb training contains 25,000 labeled reviews.

**For the final assignment, keep these as `None`** if you have a GPU and enough time.

If you are running on a CPU and only want to test whether the notebook works, temporarily use values such as:

```python
MAX_TRAIN_SAMPLES = 3000
MAX_VAL_SAMPLES = 500
MAX_TEST_SAMPLES = 1000
```

After confirming that everything works, set them back to `None` for the complete experiment.

In [ ]:
MAX_TRAIN_SAMPLES = None
MAX_VAL_SAMPLES = None
MAX_TEST_SAMPLES = None

def limit_data(texts, labels, max_samples, seed=42):
    if max_samples is None or max_samples >= len(texts):
        return texts, labels

    rng = np.random.default_rng(seed)
    indices = rng.choice(len(texts), size=max_samples, replace=False)
    indices = sorted(indices)

    return [texts[i] for i in indices], [labels[i] for i in indices]

train_texts, train_labels = limit_data(
    train_texts, train_labels, MAX_TRAIN_SAMPLES
)

val_texts, val_labels = limit_data(
    val_texts, val_labels, MAX_VAL_SAMPLES
)

test_texts, test_labels = limit_data(
    test_texts, test_labels, MAX_TEST_SAMPLES
)

print("Final training samples:", len(train_texts))
print("Final validation samples:", len(val_texts))
print("Final test samples:", len(test_texts))

## 8. Load Pretrained BERT Tokenizer

We use:

**`bert-base-uncased`**

BERT converts text into tokens and then into numerical IDs that can be processed by the neural network.

In [ ]:
MODEL_NAME = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Tokenizer loaded successfully.")
print("Vocabulary size:", tokenizer.vocab_size)

## 9. Tokenize the Reviews

`max_length=256` limits each review to 256 tokens.

Longer reviews are truncated and shorter reviews are padded when required.

In [ ]:
class IMDbDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=256):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )

        item = {key: value.squeeze(0) for key, value in encoding.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)

        return item


MAX_LENGTH = 256

train_dataset = IMDbDataset(
    train_texts, train_labels, tokenizer, MAX_LENGTH
)

val_dataset = IMDbDataset(
    val_texts, val_labels, tokenizer, MAX_LENGTH
)

test_dataset = IMDbDataset(
    test_texts, test_labels, tokenizer, MAX_LENGTH
)

print("Dataset objects created successfully.")

In [ ]:
# See how BERT tokenizes one review
sample_text = train_texts[0]

tokens = tokenizer.tokenize(sample_text[:500])
token_ids = tokenizer.encode(
    sample_text[:500],
    truncation=True,
    max_length=20
)

print("Original text:")
print(sample_text[:500])

print("\nFirst tokens:")
print(tokens[:30])

print("\nToken IDs:")
print(token_ids)

## 10. Load Pretrained BERT for Binary Classification

The pretrained BERT model receives a classification head with **2 output classes**:

```text
0 → Negative
1 → Positive
```

We then fine-tune BERT on the IMDb reviews.

In [ ]:
id2label = {
    0: "NEGATIVE",
    1: "POSITIVE"
}

label2id = {
    "NEGATIVE": 0,
    "POSITIVE": 1
}

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label=id2label,
    label2id=label2id
)

print("BERT model loaded.")

## 11. Evaluation Metrics

We calculate:

- Accuracy
- Precision
- Recall
- F1-score

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    accuracy = accuracy_score(labels, predictions)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="binary",
        zero_division=0
    )

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

## 12. Configure BERT Training

The settings below are suitable for a normal BERT fine-tuning experiment.

If you have a GPU, training will be significantly faster.

For a CPU, start with the smaller dataset settings above to verify the notebook.

In [ ]:
OUTPUT_DIR = "./bert_imdb_model"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    logging_strategy="steps",
    logging_steps=100,
    save_total_limit=2,
    report_to="none",
    fp16=torch.cuda.is_available()
)

print("Training configuration created.")

## 13. Create Trainer

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

print("Trainer ready.")

## 14. Fine-Tune BERT

**This is the main training cell.**

Depending on your computer/GPU, full IMDb training can take some time.

In [ ]:
train_result = trainer.train()

print("Training completed.")

## 15. Save the Fine-Tuned Model

In [ ]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("Model saved to:", os.path.abspath(OUTPUT_DIR))

## 16. Evaluate on the Validation Set

In [ ]:
val_results = trainer.evaluate(eval_dataset=val_dataset)

print("Validation Results:")
for key, value in val_results.items():
    if isinstance(value, (int, float)):
        print(f"{key}: {value:.4f}")

## 17. Final Evaluation on the IMDb Test Set

The official IMDb test set is used here for the final performance measurement.

In [ ]:
test_results = trainer.evaluate(eval_dataset=test_dataset)

print("Test Results:")
for key, value in test_results.items():
    if isinstance(value, (int, float)):
        print(f"{key}: {value:.4f}")

## 18. Detailed Classification Report

In [ ]:
pred_output = trainer.predict(test_dataset)

test_logits = pred_output.predictions
test_predictions = np.argmax(test_logits, axis=-1)

print(classification_report(
    test_labels,
    test_predictions,
    target_names=["Negative", "Positive"],
    digits=4
))

## 19. Confusion Matrix

In [ ]:
cm = confusion_matrix(test_labels, test_predictions)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    xticklabels=["Negative", "Positive"],
    yticklabels=["Negative", "Positive"]
)

plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")
plt.title("BERT Confusion Matrix — IMDb Test Set")
plt.show()

## 20. Calculate Final Metrics

In [ ]:
accuracy = accuracy_score(test_labels, test_predictions)

precision, recall, f1, _ = precision_recall_fscore_support(
    test_labels,
    test_predictions,
    average="binary",
    zero_division=0
)

final_metrics = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1 Score"],
    "Score": [accuracy, precision, recall, f1]
})

final_metrics["Score"] = final_metrics["Score"].round(4)

display(final_metrics)

## 21. Test BERT on New Sentences

You can enter your own movie-review sentence and see whether BERT predicts it as positive or negative.

In [ ]:
def predict_sentiment(text):
    model.eval()

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=MAX_LENGTH
    )

    device = model.device
    inputs = {key: value.to(device) for key, value in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    probabilities = torch.softmax(outputs.logits, dim=-1)
    predicted_class = torch.argmax(probabilities, dim=-1).item()
    confidence = probabilities[0][predicted_class].item()

    sentiment = "POSITIVE" if predicted_class == 1 else "NEGATIVE"

    return sentiment, confidence


examples = [
    "This movie was absolutely fantastic. I loved every minute of it.",
    "The movie was boring, poorly written, and a complete waste of time.",
    "The acting was excellent and the story was very entertaining."
]

for text in examples:
    sentiment, confidence = predict_sentiment(text)
    print("Review:", text)
    print(f"Prediction: {sentiment} ({confidence:.2%})")
    print("-" * 80)

## 22. Interactive Prediction

Change `my_review` and run the cell to test your own review.

In [ ]:
my_review = "The movie had a great story and amazing acting."

sentiment, confidence = predict_sentiment(my_review)

print("Review:", my_review)
print("Predicted Sentiment:", sentiment)
print(f"Confidence: {confidence:.2%}")

# Conclusion

### Objective
To perform sentiment analysis on IMDb movie reviews using a pretrained **BERT** model.

### Method
1. Loaded the IMDb movie review dataset.
2. Assigned `positive = 1` and `negative = 0`.
3. Split the training data into training and validation sets.
4. Used the BERT tokenizer to convert reviews into token IDs.
5. Loaded pretrained `bert-base-uncased`.
6. Fine-tuned BERT for binary sentiment classification.
7. Evaluated the model using Accuracy, Precision, Recall and F1-score.
8. Generated a confusion matrix.
9. Tested the trained model on new movie-review sentences.

### Final Result
The final Accuracy, Precision, Recall and F1-score are displayed in the **Final Metrics** table above.

> **Viva point:** BERT is a Transformer-based pretrained language model that uses bidirectional contextual representations. For sentiment analysis, BERT is fine-tuned by adding a classification head and training it on labeled reviews.